In [1]:
import torch
from torch import nn

## Self Attention Block

In [15]:
class Attention(nn.Module):
    def __init__(self,input_size,attention_size):
        super().__init__()
        self.input_size=input_size
        self.attention_size=attention_size
        self.q=nn.Linear(self.input_size,self.attention_size)
        self.k=nn.Linear(self.input_size,self.attention_size)
        self.v=nn.Linear(self.input_size,self.attention_size)
    def forward(self,x):
        key=self.k(x)
        query=self.q(x)
        value=self.v(x)
        score=query@key.transpose(-2,-1)
        scaled_score=score/torch.sqrt(torch.tensor(key.size(-1), dtype=torch.float32))
        softmax_score=torch.softmax(scaled_score,dim=-1)
        attention=softmax_score@value
        return attention

In [16]:
x=torch.rand((1,5,5))
model=Attention(5,5)
result=model(x)
print(result)

tensor([[[ 0.6537, -0.1528, -0.2359,  0.1769, -0.2731],
         [ 0.6543, -0.1520, -0.2371,  0.1763, -0.2735],
         [ 0.6541, -0.1526, -0.2360,  0.1769, -0.2731],
         [ 0.6553, -0.1514, -0.2382,  0.1755, -0.2738],
         [ 0.6539, -0.1533, -0.2348,  0.1773, -0.2724]]],
       grad_fn=<UnsafeViewBackward0>)


## Multi-Head Attention Block Architechture

### Multi Head Model

In [22]:
class Block(nn.Module):
    def __init__(self,input_size,number_heads):
        super().__init__()
        self.number_heads=number_heads
        self.block_size=input_size//number_heads
        self.q=nn.Linear(input_size,input_size)
        self.k=nn.Linear(input_size,input_size)
        self.v=nn.Linear(input_size,input_size)
        self.output=nn.Linear(input_size,input_size)
    def forward(self,x):
        batch_size,sequence_size,input_size=x.shape
        query=self.q(x)
        key=self.k(x)
        value=self.v(x)
        query=query.view(batch_size,sequence_size,self.number_heads,self.block_size).transpose(1,2)
        key=key.view(batch_size,sequence_size,self.number_heads,self.block_size).transpose(1,2)
        value=value.view(batch_size,sequence_size,self.number_heads,self.block_size).transpose(1,2)
        score=query@key.transpose(-1,-2)
        score=score/(self.block_size**0.5)
        score=torch.softmax(score,dim=-1)
        attention=score@value
        attention=attention.contiguous().view(batch_size,sequence_size,input_size)
        return self.output(attention)

### Model execution

In [23]:
x=torch.randn(1,5,8)
model=Block(input_size=8,number_heads=2)
output=model(x)
print(output.shape)

torch.Size([1, 5, 8])
